# 01-Nowicki: 数据读入 + 质量控制

> 作者已完成 QC + normalization。本 notebook 仅做数据读入、格式对齐与基础标记，不做任何过滤。

## 参数（唯一入口）

本 notebook 全部可调参数集中在此四组，改完从头 Run All；每次调参必须改 RUN_ID，禁止覆盖旧 run。

### 组1 · 数据与版本

In [ ]:
# 数据源 manifest（YAML）相对项目根路径
MANIFEST_PATH = "data/nowicki/manifest.yaml"

### 组2 · 科学参数（QC 策略 + doublet 阈值）

这些参数直接改变科学结论，每个都附四要素注释：含义 / 默认依据 / 调大调小影响 / 何时该改。

In [ ]:
# ===== 科学参数：QC 策略 =====

# 含义：QC 过滤策略，控制是否执行基本过滤（基因数/UMI/mito% 等阈值）
# 默认依据：nowicki 作者已完成 basic_filter+doublet_removal+normalization（manifest qc_overrides 全 skip），
#   重复过滤是科学错误
# 调大调小：改为非 skip 会声明要重新过滤（当前 notebook 无重滤分支，等于无操作）
# 何时该改：仅当换用未经作者 QC 的原始文件时
QC_STRATEGY = "skip"

# ===== 科学参数：细胞周期评分 =====

# 含义：是否计算 S/G2M 细胞周期评分，写入 obs["phase"] 列
# 默认依据：细胞周期是常见混杂因素，默认打开供下游回归
# 调大调小：False 则跳过、obs 无 phase 列
# 何时该改：确定不需回归细胞周期时设 False（数据用 ensembl 无法匹配 symbol 时 cell 已自动 try/except 降级）
SCORE_CELL_CYCLE = True

# ===== 科学参数：doublet 期望率 =====

# 含义：scrublet 期望双细胞率，None 表示依赖 manifest skip 判定
# 默认依据：nowicki 作者已去 doublet，默认走跳过分支
# 调大调小：>0.10 过度标记 / <0.02 漏检
# 何时该改：仅当改 manifest 移除 doublet_removal 并要本框架重跑 scrublet 时，
#   设为平台经验值（10x 常用 0.05–0.08）
EXPECTED_DOUBLET_RATE = None

# ===== 科学参数：doublet 手动阈值 =====

# 含义：手动覆盖 scrublet 自动阈值上界；None 为自动
# 默认依据：None 用 scrublet 双峰自动阈值，最稳
# 调大调小：调高漏标双细胞 / 调低误标正常细胞
# 何时该改：先跑一次看直方图，自动阈值明显落错位置时手动定
DOUBLET_SCORE_THRESHOLD = None

# ===== 科学参数：uncertain 带宽比例 =====

# 含义：uncertain 带宽 = threshold * margin；score 处于 [threshold*(1-margin), threshold] 区间为 uncertain
# 默认依据：阈值下方 20% 作经验缓冲区
# 调大调小：增大 → 更多细胞进 uncertain（保守，保留稀有亚群）/ 减小 → 更少 uncertain
# 何时该改：怀疑高复杂度真实细胞（激活免疫细胞/浆细胞）被误判 doublet 时增大
DOUBLET_UNCERTAIN_MARGIN = 0.2

# ===== 科学参数：doublet 最小细胞数 =====

# 含义：样本细胞数下界，低于此不定阈值、全标 singlet、触发 needs_review
# 默认依据：scrublet 官方下界（<50 无法可靠估计 score 双峰）
# 调大调小：增大 → 更多小样本进 needs_review（保守）/ 减小 → 冒险给小样本定阈值
# 何时该改：样本普遍偏小且已知双细胞率低时可适度下调
DOUBLET_MIN_CELLS = 50

# ===== 科学参数：doublet 比例警戒阈 =====

# 含义：预测 doublet 比例警戒阈，超过触发 needs_review
# 默认依据：0.30 提示解离过度/活性差/混样
# 调大调小：增大 → 更少样本被 flag（宽松）/ 减小 → 更多样本进 needs_review（严格）
# 何时该改：已知某平台双细胞率系统偏高时适度上调
DOUBLET_RATE_ALERT = 0.30

### 组3 · 计算参数

In [ ]:
# 随机种子，保证 scrublet / 降维等可复现
RANDOM_SEED = 42

### 组4 · 输出与运行标识

In [ ]:
# 每次调参改新 ID，禁止覆盖旧 run
RUN_ID = "01-nowicki-v1-run001"
# run 产物根目录
RUN_ROOT = "results/runs"
# 输出文件名
OUTPUT_FILENAME = "01_nowicki_v1.h5ad"
# 输出版本号，写入 uns["version"]=f"v{OUTPUT_VERSION}"
OUTPUT_VERSION = 1

In [ ]:
# === Setup：sys.path + 导入依赖 ===
import sys, os
_root = os.getcwd()
# 向上逐级查找项目根（含 src/scrna_integration 的目录），兼容任意嵌套深度
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
        break
    _root = os.path.dirname(_root)
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# A800 64核 OpenBLAS默认全开致线程爆炸（200+线程冻结），限制为4
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMBA_NUM_THREADS", "4")

import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc
from pathlib import Path
from scrna_integration.run_contract import (
    atomic_write_json, collect_runtime_provenance, determine_stage_status, prepare_run,
    promote_run, sha256_file, snapshot_effective_parameters, validate_expression_contract,
)

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

import importlib.metadata
print(f"Scanpy {importlib.metadata.version('scanpy')}  |  anndata {importlib.metadata.version('anndata')}")

from scrna_integration.io import sync_gene_ids


## Preflight 校验（数据加载前）

集中校验输入文件存在、关键参数合法、数据集特有前提（.raw.X）与 SoupX/doublet 开关一致，任一不合法立即 raise，避免带病跑完整条链。

In [ ]:
# Preflight 校验（数据加载前 fail-fast）
# 所有临时变量皆 _ 前缀，避免被 checkpoint snapshot_effective_parameters 捕获进 run manifest
import os
import yaml
import anndata as _ad

# ---- 1. manifest 路径存在 ----
if not os.path.exists(MANIFEST_PATH):
    raise FileNotFoundError(f"manifest 文件不存在: {MANIFEST_PATH}")

with open(MANIFEST_PATH) as _f:
    _mani = yaml.safe_load(_f)

# ---- 2. 输入文件存在 ----
_input_path = _mani["input"]["path"]
if not os.path.exists(_input_path):
    raise FileNotFoundError(f"输入 h5ad 文件不存在: {_input_path}")

# ---- 3. 关键 PARAMS 类型/取值合法 ----
_errs = []

if not isinstance(RANDOM_SEED, int):
    _errs.append(f"RANDOM_SEED 应为 int，实际 {type(RANDOM_SEED).__name__}")
if not isinstance(OUTPUT_VERSION, int):
    _errs.append(f"OUTPUT_VERSION 应为 int，实际 {type(OUTPUT_VERSION).__name__}")

if QC_STRATEGY not in {"skip"}:
    _errs.append(f"QC_STRATEGY 应为 'skip'（nowicki 当前只支持 skip），实际 {QC_STRATEGY!r}")

if not isinstance(SCORE_CELL_CYCLE, bool):
    _errs.append(f"SCORE_CELL_CYCLE 应为 bool，实际 {type(SCORE_CELL_CYCLE).__name__}")

if not isinstance(DOUBLET_UNCERTAIN_MARGIN, (int, float)) or not (0 < DOUBLET_UNCERTAIN_MARGIN < 1):
    _errs.append(f"DOUBLET_UNCERTAIN_MARGIN 应为 float 且 0<值<1，实际 {DOUBLET_UNCERTAIN_MARGIN!r}")

if not isinstance(DOUBLET_MIN_CELLS, int) or DOUBLET_MIN_CELLS <= 0:
    _errs.append(f"DOUBLET_MIN_CELLS 应为 int > 0，实际 {DOUBLET_MIN_CELLS!r}")

if not isinstance(DOUBLET_RATE_ALERT, (int, float)) or not (0 < DOUBLET_RATE_ALERT <= 1):
    _errs.append(f"DOUBLET_RATE_ALERT 应为 float 且 0<值<=1，实际 {DOUBLET_RATE_ALERT!r}")

# EXPECTED_DOUBLET_RATE: None 或 (float 且 0<值<1)
_edr = EXPECTED_DOUBLET_RATE
if _edr is not None:
    if not isinstance(_edr, (int, float)) or not (0 < _edr < 1):
        _errs.append(f"EXPECTED_DOUBLET_RATE 应为 None 或 float 且 0<值<1，实际 {_edr!r}")

# DOUBLET_SCORE_THRESHOLD: None 或 (float 且 >0)
_dst = DOUBLET_SCORE_THRESHOLD
if _dst is not None:
    if not isinstance(_dst, (int, float)) or _dst <= 0:
        _errs.append(f"DOUBLET_SCORE_THRESHOLD 应为 None 或 float > 0，实际 {_dst!r}")

if _errs:
    raise ValueError("Preflight 参数校验失败:\n" + "\n".join(f"  - {e}" for e in _errs))

# ---- 4. SoupX 禁用断言（nowicki 无完整 raw droplets，决策3 禁用）----
if globals().get("SOUPX_ENABLED", False):
    raise ValueError("Nowicki 无完整 raw droplets，不支持 SoupX（决策3）")

# ---- 5. .raw.X 必须存在（nowicki X 是 logcounts，counts 只在 .raw.X）----
_probe = _ad.read_h5ad(_input_path, backed="r")
if _probe.raw is None:
    _probe.file.close()
    raise ValueError("Nowicki 数据必须包含 .raw.X 原始计数，preflight 未在输入文件探测到 .raw")
_probe.file.close()
del _probe

# ---- 6. doublet 配置连贯性（只 print WARNING，不 raise）----
_pp_done = _mani.get("preprocessing_done", [])
_qc_dbl = _mani.get("qc_overrides", {}).get("doublet_removal", {})
_author_did_doublet = "doublet_removal" in _pp_done or _qc_dbl.get("skip")
if not _author_did_doublet and EXPECTED_DOUBLET_RATE is None:
    print("[WARNING] 未声明作者去 doublet 且 EXPECTED_DOUBLET_RATE=None → doublet cell 将进入 needs_review（配置不完整）")

# ---- 7. 生效参数摘要 ----
_source_ds = _mani.get("source_dataset", "?")
print("=== Preflight 校验通过：生效参数摘要 ===")
print(f"  输入文件: {_input_path}")
print(f"  source_dataset: {_source_ds}")
print(f"  RUN_ID: {RUN_ID}")
print(f"  OUTPUT_VERSION: {OUTPUT_VERSION}")
print(f"  QC_STRATEGY: {QC_STRATEGY}")
print(f"  SCORE_CELL_CYCLE: {SCORE_CELL_CYCLE}")
print(f"  EXPECTED_DOUBLET_RATE: {EXPECTED_DOUBLET_RATE}")
print(f"  DOUBLET_SCORE_THRESHOLD: {DOUBLET_SCORE_THRESHOLD}")
print(f"  DOUBLET_UNCERTAIN_MARGIN: {DOUBLET_UNCERTAIN_MARGIN}")
print(f"  DOUBLET_MIN_CELLS: {DOUBLET_MIN_CELLS}")
print(f"  DOUBLET_RATE_ALERT: {DOUBLET_RATE_ALERT}")
print(f"  SoupX: 禁用")
print(f"  .raw.X: 存在")
print("========================================")

In [ ]:
# 数据读入：h5ad 格式（Nowicki 2023，ensembl 基因，已 QC + normalized）
# 替代原来的 read_with_manifest / 按透明性铁律，数据读取逻辑拆回 cell
import yaml
from scrna_integration.io import sync_gene_ids

with open(MANIFEST_PATH) as f:
    manifest = yaml.safe_load(f)
source_dataset = str(manifest["source_dataset"])

# ---- 1. 读取 h5ad ----
path = manifest["input"]["path"]
adata = sc.read_h5ad(path)
print(f"原始文件: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")

# ---- 2. obs_mapping：列重命名（作者原始列名 → 框架统一列名）----
obs_map = manifest.get("obs_mapping", {})
for target_col, source_col in obs_map.items():
    if source_col in adata.obs.columns:
        adata.obs[target_col] = adata.obs[source_col]
    else:
        print(f"  WARNING: obs_mapping 源列 '{source_col}' 不存在，跳过")
# 删除已被映射的原始列（避免下游混淆）
mapped_cols = [c for c in obs_map.values() if c in adata.obs.columns]
if mapped_cols:
    adata.obs.drop(columns=mapped_cols, inplace=True)
print(f"obs_mapping 已应用 ({len(obs_map)} 个字段)")

# ---- 3. 数据标识字段 ----
adata.obs["source_dataset"] = source_dataset
adata.obs["project_id"] = manifest.get("project_id", "")
adata.obs["disease_system"] = manifest.get("disease_system", "")

# ---- 4. Layer 2 字段填充（缺失的填 NaN）----
layer2_fields = ["disease", "disease_ontology_term_id", "tissue",
                 "tissue_ontology_term_id", "assay", "sex", "development_stage"]
for field in layer2_fields:
    if field not in adata.obs.columns:
        print(f"  [Layer2] 列 '{field}' 缺失，填充 NaN")
        adata.obs[field] = np.nan
    else:
        col = adata.obs[field]
        n_null = col.isna().sum()
        n_empty = (col.astype(str).str.strip() == "").sum()
        if n_null + n_empty > 0:
            print(f"  [Layer2] 列 '{field}' 有 {n_null + n_empty}/{len(col)} 个缺失或空值")

# ---- 5. Layer 1 确定性校验 ----
layer1_required = ["source_dataset", "project_id", "disease_system"]
missing_l1 = [f for f in layer1_required if f not in adata.obs.columns]
if missing_l1:
    raise ValueError(f"Layer 1 必需字段缺失: {missing_l1}")

# ---- 6. 基线 QC 指标 ----
if not sp.issparse(adata.X):
    adata.X = sp.csr_matrix(adata.X)
adata.obs["n_genes"] = (adata.X > 0).sum(axis=1).A1 if sp.issparse(adata.X) else (adata.X > 0).sum(axis=1)
adata.obs["total_counts"] = np.asarray(adata.X.sum(axis=1)).flatten()
mt_mask = adata.var.index.str.startswith("MT-")
if mt_mask.any():
    adata.obs["pct_counts_mt"] = (
        np.asarray(adata.X[:, mt_mask].sum(axis=1)).flatten()
        / adata.obs["total_counts"].values * 100
    )
ribo_mask = adata.var.index.str.startswith(("RPS", "RPL"))
if ribo_mask.any():
    adata.obs["pct_counts_ribo"] = (
        np.asarray(adata.X[:, ribo_mask].sum(axis=1)).flatten()
        / adata.obs["total_counts"].values * 100
    )

# ---- 7. 基因 ID 同步：Ensembl → Symbol ----
# Nowicki 数据 var.index 为 Ensembl ID（ENSG...），需转为 gene symbol
# 这是 02_merged inner join 的前提——所有数据集的 var.index 必须统一为 symbol
sync_gene_ids(adata, gene_id_format="ensembl")

# ---- 8. 从 .raw.X 提取 counts，写 layers["counts"] ----
# Nowicki: X 已是 logcounts，raw counts 在 .raw.X。
# 提取整数 counts、校验非负整数 + shape 对齐后写入 layers["counts"]（CSR float32）。
if adata.raw is None:
    raise ValueError("Nowicki 数据必须包含 .raw.X 原始计数，未找到 .raw 属性")

raw_adata = adata.raw.to_adata()

# 基因维度不一致检查：.raw.X 的 var 必须与当前 adata.var 完全对齐，不一致直接 raise
if list(raw_adata.var_names) != list(adata.var_names):
    raise ValueError(
        f"Nowicki .raw.X 基因维度与当前 adata 不一致："
        f"raw {raw_adata.n_vars} genes vs current {adata.n_vars} genes。"
        f"基因维度不一致，停止处理。请检查 manifest 使用的文件是否完整。"
    )

# 提取 counts 矩阵
counts_matrix = raw_adata.X.copy()

# 校验非负整数
counts_data = counts_matrix.data if sp.issparse(counts_matrix) else np.asarray(counts_matrix).ravel()
if counts_data.size == 0:
    raise ValueError("counts matrix is empty")
if (counts_data < 0).any():
    raise ValueError("counts contain negative values")
if not np.allclose(counts_data % 1, 0):
    raise ValueError("counts contain non-integer values")

# 写入 layers["counts"]（CSR float32）
adata.layers["counts"] = sp.csr_matrix(counts_matrix, dtype=np.float32)

# 写入 expression_contract
adata.uns["expression_contract"] = {
    "x_scale": "normalized_log1p",
    "counts_layer": "counts",
    "counts_source": ".raw.X",
    "counts_validated": True,
    "counts_integer_check": "full",
    "soupx_layer": None,
    "processing_history": ["01_extract_counts_from_raw"],
    "stage": "01",
}
print(f"layers['counts'] 已建立: {adata.layers['counts'].shape}，来自 .raw.X")
print(f"expression_contract: x_scale=normalized_log1p, counts_source=.raw.X, stage=01")

# ---- 9. 记录预处理状态（供下游 03_normalized 读取）----
adata.uns["preprocessing_done"] = manifest.get("preprocessing_done", [])
adata.uns["qc_overrides"] = manifest.get("qc_overrides", {})

print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"obs 列: {list(adata.obs.columns)}")
print(f"var 列: {list(adata.var.columns)}")


## 双细胞鉴定（三态政策）

决策8 三态分类策略落地 per-dataset 01 notebook：

- **singlet**（正常单细胞）：score 低于 uncertain 下界，纳入下游分析
- **uncertain**（不确定）：score 处于阈值下方 uncertain 带宽内，保留供下游 04/05 观察
- **doublet**（高置信双细胞）：score 高于阈值，`doublet_include=False`，默认不在 02 整合对象中出现

**为什么按样本独立定阈值**：不同样本的细胞量、RNA 质量、解离条件不同，
双细胞率也不同。合并定阈值会将高复杂度真实细胞（如激活免疫细胞、浆细胞）
误判为 doublet，或让低质量样本的 doublet 漏过。

**为什么只排除高置信 doublet 而保留 uncertain**：uncertain 带内的细胞可能
是真实稀有亚群（高 UMI 计数 + 高基因数导致 score 偏高），贸然删除可能
丢失生物学信号。下游 04/05 持续观察 uncertain 是否形成独立群再做判断。

**nowicki 特殊性**：作者已完成 doublet_removal，默认走 `author_removed`
跳过分支，但四列 + `uns["doublet_contract"]` 仍统一写全，
确保下游 04/05 接口一致。若 PI 改 manifest 去掉 doublet_removal
且设 `EXPECTED_DOUBLET_RATE` 非 None，则走 scrublet 分支。

**与 Kim 的数据差异**：nowicki 的 `X` 是 logcounts，scrublet 输入
必须取 `layers["counts"]`（整数 counts）；Kim 的 `X` 本身即 counts，
直接用 `sub.X`。这是两个数据集在 doublet 检测上的唯一实质差异。


In [ ]:
# 双细胞鉴定：三态分类 + doublet_include + needs_review
#
# 决策8（三态政策）落地到 per-dataset 01 notebook：
# - doublet_class 三态：singlet（正常）| uncertain（不确定）| doublet（高置信双细胞）
# - 仅高置信 doublet 标 doublet_include=False，默认排入下游02整合对象
# - uncertain 保留（include=True），供下游04/05观察——若形成独立群则可能是真实稀有亚群
# - 按样本独立定阈值，因为不同样本的细胞量、RNA质量、双细胞率不同
#
# nowicki 特殊性：作者已完成 doublet_removal，默认走 author_removed 跳过分支。
# 跳过时仍统一记录四列 + uns["doublet_contract"]，确保下游 04/05 接口一致。
#
# 若 PI 在 manifest 中移除 doublet_removal 并设 EXPECTED_DOUBLET_RATE 非 None，
# 则走 scrublet 分支。nowicki 的 X 是 logcounts，scrublet 必须取 layers["counts"]（整数counts），
# 严禁用 sub.X——这与 Kim（X 本身即 counts）不同。

# --- 三态分类函数 ---
# 边界语义（与测试 test_p1c_doublet_nowicki.py 逐字一致）：
#   score > threshold                                        → doublet
#   (1 - margin) * threshold < score <= threshold             → uncertain
#   score <= (1 - margin) * threshold                         → singlet
# uncertain 带宽 = threshold * margin：score 处于阈值下方这个区间内的细胞，
# 可能是低质量双细胞，也可能是高复杂度真实细胞（如激活的免疫细胞、浆细胞）。
# 下游 04/05 持续观察这些 uncertain 细胞是否形成独立群再做判断。
def classify_doublets(scores, threshold, margin):
    """将 doublet scores 按三态策略分类。

    Parameters
    ----------
    scores : np.ndarray
        scrublet 输出的 doublet scores
    threshold : float
        上界阈值（scrublet 自动阈值或手动覆盖值）
    margin : float
        uncertain 带宽比例（threshold * margin 为 uncertain 带宽度）

    Returns
    -------
    np.ndarray of str: "singlet" | "uncertain" | "doublet"
    """
    result = np.full(len(scores), "singlet", dtype=object)
    lower_bound = (1 - margin) * threshold
    result[scores > threshold] = "doublet"
    result[(scores > lower_bound) & (scores <= threshold)] = "uncertain"
    return result


# --- 初始化四列：无论哪个分支，最终都有这四列 ---
# 用显式标记变量控制首次初始化（避免依赖 dir(adata.obs) 判断结果残留）
_DOUBLET_COLS_INITIALIZED = False
if "doublet_score" not in adata.obs.columns:
    adata.obs["doublet_score"] = np.nan
if "doublet_class" not in adata.obs.columns:
    adata.obs["doublet_class"] = "singlet"
if "doublet_include" not in adata.obs.columns:
    adata.obs["doublet_include"] = True
if "predicted_doublet" not in adata.obs.columns:
    adata.obs["predicted_doublet"] = False
_DOUBLET_COLS_INITIALIZED = True

# 确保类型正确（idempotent——重复运行 cell 也不会破坏类型）
adata.obs["doublet_score"] = adata.obs["doublet_score"].astype(np.float32)
adata.obs["doublet_class"] = adata.obs["doublet_class"].astype(object)
adata.obs["doublet_include"] = adata.obs["doublet_include"].astype(bool)
adata.obs["predicted_doublet"] = adata.obs["predicted_doublet"].astype(bool)


# --- 判定是否跳过双细胞鉴定 ---
with open(MANIFEST_PATH) as f:
    _dm = yaml.safe_load(f)
_pp_done = _dm.get("preprocessing_done", [])
_qc_dbl = _dm.get("qc_overrides", {}).get("doublet_removal", {})

skip_doublet = False
doublet_method = None
skip_reason = None

if "doublet_removal" in _pp_done:
    skip_doublet = True
    doublet_method = "author_removed"
    skip_reason = "原作者已去除双细胞（preprocessing_done 含 doublet_removal）"
elif _qc_dbl.get("skip"):
    skip_doublet = True
    doublet_method = "author_removed"
    skip_reason = _qc_dbl.get("reason", "qc_overrides.doublet_removal.skip=True")

# 是否应运行 scrublet（跳过为 True 或 EXPECTED_DOUBLET_RATE 为 None 则不运行）
_should_run_scrublet = (not skip_doublet) and (EXPECTED_DOUBLET_RATE is not None)

# --- 诊断容器 ---
per_sample_thresholds = {}
per_sample_diagnostics = {}
review_reasons = []
doublet_needs_review = False


if skip_doublet:
    # =====================================================================
    # 跳过分支（nowicki 默认路径）：manifest 明确声明作者已去 doublet
    # =====================================================================
    print(f"双细胞鉴定已跳过: {skip_reason}")
    # 四列已初始化：doublet_score=NaN, doublet_class="singlet", doublet_include=True, predicted_doublet=False
    _n_singlet = adata.n_obs
    _n_uncertain = 0
    _n_doublet = 0
    _n_excluded = 0  # 无不纳入的细胞
    adata.uns["doublet_contract"] = {
        "method": doublet_method,
        "version": "manifest_skip",
        "per_sample_thresholds": {},
        "uncertain_margin": float(DOUBLET_UNCERTAIN_MARGIN),
        "expected_doublet_rate": None,
        "skip_reason": skip_reason,
        "n_singlet": int(_n_singlet),
        "n_uncertain": int(_n_uncertain),
        "n_doublet": int(_n_doublet),
        "n_excluded": int(_n_excluded),
        "needs_review": False,
        "review_reasons": [],
        "per_sample_diagnostics": {},
    }
    print(f"  singlet={_n_singlet}  uncertain={_n_uncertain}  doublet={_n_doublet}  excluded={_n_excluded}")

elif _should_run_scrublet:
    # =====================================================================
    # scrublet 分支：仅当 PI 改 manifest 移除 doublet_removal 且设 EXPECTED_DOUBLET_RATE 非 None
    # =====================================================================
    import scrublet as scr
    import importlib.metadata

    print(f"运行 Scrublet per sample (expected_doublet_rate={EXPECTED_DOUBLET_RATE})...")
    # nowicki 特有：scrublet 输入取 layers["counts"]（整数 counts），不取 sub.X（logcounts）
    # 原因：nowicki 的 X 是 normalized_log1p，非整数；喂 logcounts 给 scrublet 会导致
    # doublet score 分布异常（双峰消失或阈值无意义），造成科学错误。
    # Kim 的 X 本身即 raw counts，所以 Kim 用 sub.X——这是两个数据集的唯一差异。
    print(f'  （nowicki: 输入矩阵取 layers["counts"] 而非 sub.X，因为 X 是 logcounts）')

    sample_ids = sorted(adata.obs["sample_id"].unique())

    for sample_id in sample_ids:
        mask = adata.obs["sample_id"] == sample_id
        n_sample = mask.sum()

        # 样本细胞数不足 → flagged + 不定阈值 + 全标 singlet
        if n_sample < DOUBLET_MIN_CELLS:
            msg = f"样本细胞数 {n_sample} < DOUBLET_MIN_CELLS={DOUBLET_MIN_CELLS}，不定阈值"
            print(f"  {sample_id}: {msg}，所有细胞标 singlet")
            per_sample_thresholds[sample_id] = None
            per_sample_diagnostics[sample_id] = {
                "n_cells": int(n_sample), "n_doublet": 0, "pct_doublet": 0.0,
                "threshold": None, "flagged": True, "flag_reason": msg,
            }
            review_reasons.append(f"{sample_id}: {msg}")
            doublet_needs_review = True
            continue

        # 取样本子集（copy 后即释放，避免内存堆积）
        sub = adata[mask].copy()

        # nowicki 特有：scrublet 输入取 layers["counts"]（整数 counts），不取 sub.X
        # （详见上文注释；与 Kim 的差异点）
        counts_for_scrublet = sub.layers["counts"]

        scrub = scr.Scrublet(
            counts_for_scrublet,
            expected_doublet_rate=EXPECTED_DOUBLET_RATE,
            random_state=RANDOM_SEED,
        )
        doublet_scores, _predicted = scrub.scrub_doublets()

        # 确定阈值：手动覆盖优先，否则取 scrublet 自动阈值
        manual_thresh = DOUBLET_SCORE_THRESHOLD
        threshold = float(manual_thresh) if manual_thresh is not None else scrub.threshold_
        if threshold is None or not np.isfinite(threshold):
            msg = "scrublet 阈值无法确定（None/非有限值），所有细胞标 singlet"
            print(f"  {sample_id}: {msg}")
            per_sample_thresholds[sample_id] = None
            per_sample_diagnostics[sample_id] = {
                "n_cells": int(n_sample), "n_doublet": 0, "pct_doublet": 0.0,
                "threshold": None, "flagged": True, "flag_reason": msg,
            }
            review_reasons.append(f"{sample_id}: {msg}")
            doublet_needs_review = True
            del sub; gc.collect()
            continue
        threshold = float(threshold)
        per_sample_thresholds[sample_id] = threshold

        # 三态分类
        states = classify_doublets(doublet_scores, threshold, DOUBLET_UNCERTAIN_MARGIN)
        n_dbl = int((states == "doublet").sum())
        n_unc = int((states == "uncertain").sum())
        n_sgl = int((states == "singlet").sum())
        pct_dbl = n_dbl / n_sample if n_sample > 0 else 0.0

        # 写回 adata.obs
        adata.obs.loc[mask, "doublet_score"] = doublet_scores.astype(np.float32)
        adata.obs.loc[mask, "doublet_class"] = states
        adata.obs.loc[mask, "doublet_include"] = (states != "doublet")  # 仅高置信 doublet 排除
        adata.obs.loc[mask, "predicted_doublet"] = (states == "doublet")  # bool 兼容列

        # per-sample 诊断与 needs_review 判定
        flagged = False
        flag_reasons = []
        if pct_dbl > DOUBLET_RATE_ALERT:
            flagged = True
            flag_msg = f"doublet 比例 {pct_dbl:.1%} > DOUBLET_RATE_ALERT={DOUBLET_RATE_ALERT:.1%}"
            flag_reasons.append(flag_msg)
            review_reasons.append(f"{sample_id}: {flag_msg}")
            doublet_needs_review = True

        per_sample_diagnostics[sample_id] = {
            "n_cells": int(n_sample), "n_doublet": n_dbl,
            "pct_doublet": float(pct_dbl), "threshold": threshold,
            "flagged": flagged,
            "flag_reason": "; ".join(flag_reasons) if flag_reasons else None,
        }

        # 直方图可视化
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(doublet_scores, bins=50, color="steelblue", edgecolor="white", alpha=0.7)
        ax.axvline(threshold, color="red", linestyle="--",
                   label=f"阈值={threshold:.3f} (doublet)")
        lower_bound = (1 - DOUBLET_UNCERTAIN_MARGIN) * threshold
        ax.axvline(lower_bound, color="orange", linestyle=":",
                   label=f"uncertain 下界={lower_bound:.3f}")
        ax.set_xlabel("Doublet Score")
        ax.set_ylabel("细胞数")
        ax.set_title(f"{sample_id}: Doublet Score 分布 (dbl={n_dbl}, unc={n_unc}, sgl={n_sgl})")
        ax.legend()
        plt.tight_layout(); plt.show()

        print(f"  {sample_id}: doublet={n_dbl}  uncertain={n_unc}  singlet={n_sgl}  "
              f"({pct_dbl:.1%} doublet)  threshold={threshold:.4f}")

        del sub; gc.collect()

    # 汇总计数
    _n_doublet = int((adata.obs["doublet_class"] == "doublet").sum())
    _n_uncertain = int((adata.obs["doublet_class"] == "uncertain").sum())
    _n_singlet = int((adata.obs["doublet_class"] == "singlet").sum())
    _n_excluded = int((~adata.obs["doublet_include"]).sum())

    # 获取 scrublet 版本
    _scrublet_version = "unknown"
    try:
        _scrublet_version = importlib.metadata.version("scrublet")
    except Exception:
        pass

    # 写入 uns 元数据
    adata.uns["doublet_contract"] = {
        "method": "scrublet",
        "version": _scrublet_version,
        "per_sample_thresholds": per_sample_thresholds,
        "uncertain_margin": float(DOUBLET_UNCERTAIN_MARGIN),
        "expected_doublet_rate": float(EXPECTED_DOUBLET_RATE)
            if EXPECTED_DOUBLET_RATE is not None else None,
        "skip_reason": None,
        "n_singlet": _n_singlet,
        "n_uncertain": _n_uncertain,
        "n_doublet": _n_doublet,
        "n_excluded": _n_excluded,
        "needs_review": doublet_needs_review,
        "review_reasons": review_reasons,
        "per_sample_diagnostics": per_sample_diagnostics,
    }

    print(f"\n双细胞鉴定完成:")
    print(f"  singlet={_n_singlet}  uncertain={_n_uncertain}  doublet={_n_doublet}  excluded={_n_excluded}")
    print(f"  needs_review={doublet_needs_review}")
    if review_reasons:
        for r in review_reasons:
            print(f"    [需要关注] {r}")
    print(f"  注意：01 只标记 doublet_include，未物理删除细胞。物理排除在 02（P1-d）执行。")

else:
    # =====================================================================
    # 未跳过但 EXPECTED_DOUBLET_RATE=None：配置不完整，标记 needs_review
    # （nowicki 默认路径下不会走到这里；仅当 PI 改 manifest 移除 doublet_removal
    #   但忘记设 EXPECTED_DOUBLET_RATE 时才会触发）
    # =====================================================================
    print("EXPECTED_DOUBLET_RATE=None 且 manifest 未声明 doublet_removal，跳过 scrublet")
    _n_singlet = adata.n_obs
    _n_uncertain = 0
    _n_doublet = 0
    _n_excluded = 0
    doublet_needs_review = True
    review_reasons.append("EXPECTED_DOUBLET_RATE=None 但 manifest 未声明 doublet_removal：是否遗漏了双细胞去除步骤？")
    adata.uns["doublet_contract"] = {
        "method": "skipped_incomplete",
        "version": "none",
        "per_sample_thresholds": {},
        "uncertain_margin": float(DOUBLET_UNCERTAIN_MARGIN),
        "expected_doublet_rate": None,
        "skip_reason": "EXPECTED_DOUBLET_RATE=None 且 manifest 无 doublet_removal：配置不完整，请检查是否需要运行 doublet 检测",
        "n_singlet": int(_n_singlet),
        "n_uncertain": int(_n_uncertain),
        "n_doublet": int(_n_doublet),
        "n_excluded": int(_n_excluded),
        "needs_review": True,
        "review_reasons": review_reasons,
        "per_sample_diagnostics": {},
    }
    print(f"  singlet={_n_singlet}  uncertain={_n_uncertain}  doublet={_n_doublet}  excluded={_n_excluded}")
    print(f"  配置不完整，已标记 needs_review=True")


## 细胞周期评分

In [ ]:
# 细胞周期评分（Tirosh 2015 marker genes）
# scanpy 默认的 S/G2M 基因是 gene symbol 格式，但部分数据集使用 ensembl ID
# 作为基因名（如 Nowicki 原始数据），导致 scanpy 找不到匹配基因而抛出
# ValueError。用 try/except 优雅降级——跳过评分并标记为 unknown，不阻断管线。
if SCORE_CELL_CYCLE:
    s_genes = ["MCM5","PCNA","TYMS","FEN1","MCM2","MCM4","RRM1","UNG","GINS2","MCM6","CDCA7","DTL","PRIM1","UHRF1","MLF1IP","HELLS","RFC2","RPA2","NASP","RAD51AP1","GMNN","WDR76","SLBP","CCNE2","UBR7","POLD3","MSH2","ATAD2","RAD51","RRM2","CDC45","CDC6","EXO1","TIPIN","DSCC1","BLM","CASP8AP2","USP1","CLSPN","POLA1","CHAF1B","BRIP1","E2F8"]
    g2m_genes = ["HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A","NDC80","CKS2","NUF2","CKS1B","MKI67","TMPO","CENPF","TACC3","FAM64A","SMC4","CCNB2","CKAP2L","CKAP2","AURKB","BUB1","KIF11","ANP32E","TUBB4B","GTSE1","KIF20B","HJURP","CDCA3","HN1","CDC20","TTK","CDC25C","KIF2C","RANGAP1","NCAPD2","DLGAP5","CDCA2","CDCA8","ECT2","KIF23","HMMR","AURKA","PSRC1","ANLN","LBR","CKAP5","CENPE","CTCF","NEK2","G2E3","GAS2L3","CBX5","CENPA"]
    try:
        sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)
        print("细胞周期评分完成")
        print(adata.obs["phase"].value_counts())
    except ValueError as e:
        # scanpy 找不到匹配的基因（如数据使用 ensembl ID 而默认基因列表是 symbol）
        # 此时无法做细胞周期评分，将对应列填充为 NaN/unknown，不阻断下游分析
        print(f"⚠️ 细胞周期评分跳过: {e}")
        print("  原因: scanpy 默认 S/G2M 基因列表为 gene symbol，数据可能使用其他 ID 体系")
        adata.obs['S_score'] = np.nan
        adata.obs['G2M_score'] = np.nan
        adata.obs['phase'] = 'unknown'
else:
    print("SCORE_CELL_CYCLE=False，跳过")


## 基因复杂度

In [ ]:
# 基因复杂度
adata.obs["log_complexity"] = np.log10(adata.obs["n_genes"] + 1) / np.log10(adata.obs["total_counts"] + 1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(adata.obs["log_complexity"].dropna(), bins=50, color="steelblue", edgecolor="white")
ax.set_xlabel("log10(n_genes+1) / log10(total_counts+1)")
ax.set_ylabel("细胞数")
ax.set_title("基因复杂度分布")
for pct in [1, 5, 25, 50, 75, 95, 99]:
    val = np.percentile(adata.obs["log_complexity"].dropna(), pct)
    ax.axvline(val, color="red", linestyle="--", alpha=0.3, linewidth=0.8)
plt.tight_layout()
fig.savefig("results/figures/01_nowicki_complexity.png", dpi=150, bbox_inches="tight")
plt.show()


## 原作者标注列确认

In [ ]:
# 确认原作者标注列已正确注入
annotation_cols = [c for c in adata.obs.columns if c.startswith("cell_type_original_")]
print("原作者标注列:")
for col in annotation_cols:
    vals = adata.obs[col].dropna().unique()
    print(f"  {col}: {len(vals)} 个唯一值: {sorted(vals)[:15]}...")


## QC 报告（跳过模式）

In [ ]:
# QC 报告
qc_report = {
    "strategy": "skip",
    "note": "作者已完成 basic_filter + doublet_removal + normalization；重新过滤会造成科学错误。",
    "cells_total": int(adata.n_obs),
    "cells_removed": 0,
    "pct_removed": 0.0,
    "cell_cycle_scored": SCORE_CELL_CYCLE,
}
adata.uns["qc_report_v1"] = qc_report
for k, v in qc_report.items():
    print(f"  {k}: {v}")


In [ ]:
# Checkpoint
import scipy.sparse as sp
# 先计算所有可见门禁；只有到保存阶段才占用 RUN_ID。

# F2修复：记录作者预处理状态到 uns，供下游 03_normalized 判断是否跳过标准化
# nowicki manifest 声明 preprocessing_done 含 "normalization"，不做记录则 03 会二次归一
import yaml
_mani_path = MANIFEST_PATH
if not os.path.isabs(_mani_path):
    _mani_path = os.path.join(_root, _mani_path)
with open(_mani_path) as _mf:
    _manifest_data = yaml.safe_load(_mf)
_pp_done = _manifest_data.get("preprocessing_done", [])
if _pp_done:
    adata.uns["preprocessing_done"] = _pp_done
    print(f"preprocessing_done 已记录: {_pp_done}")

# F3修复：基因 ID 轴统一性断言——检查 per-dataset 内基因名无大小写混用
# 过大写/小写混用在 merge 时会导致 inner join 基因交集意外坍塌
_gene_names = list(adata.var_names)
_upper_count = sum(1 for g in _gene_names if g[0].isupper()) if _gene_names else 0
_lower_count = sum(1 for g in _gene_names if g[0].islower()) if _gene_names else 0
_total = len(_gene_names)
if _upper_count > 0 and _lower_count > 0:
    raise ValueError(
        f"基因 ID 轴不一致：{_upper_count} 个大写首字母基因 + {_lower_count} 个小写首字母基因 共 {_total} 个。"
        f"请统一基因名大小写（如全部 .str.upper()）后再进入 merge。"
    )
print(f"基因 ID 轴一致性检查通过：{_total} 个基因，统一为{'大写' if _upper_count > 0 else '小写'}首字母")

_source_values = sorted(map(str, adata.obs["source_dataset"].dropna().unique())) if "source_dataset" in adata.obs.columns else []
# expression_contract 验证（决策 1/2 管线）
validate_expression_contract(adata, expected_scale="normalized_log1p")
assert adata.uns["expression_contract"]["counts_source"] == ".raw.X", (
    f"counts_source 应为 '.raw.X'，实际为 {adata.uns['expression_contract']['counts_source']!r}"
)
print("expression_contract 验证通过: counts_source=.raw.X, x_scale=normalized_log1p")

# P1-c: doublet 检测是否已运行的 guard
# 未运行 doublet 检测的场景（如 shared test 合成 adata 仅含 source_dataset 列）
# 所有 doublet 相关 postcondition 设为 True 跳过，避免 KeyError
_hd = "doublet_class" in adata.obs.columns

hard_postconditions = {
    "non_empty": adata.n_obs > 0 and adata.n_vars > 0,
    "x_sparse_float32": sp.issparse(adata.X) and adata.X.dtype == np.float32,
    "source_dataset_unique": len(_source_values) == 1 and not adata.obs["source_dataset"].isna().any(),
    "source_matches_manifest": _source_values == [str(source_dataset)],
    "counts_layer_exists": "counts" in adata.layers,
    "counts_layer_sparse": adata.layers.get("counts") is not None and sp.issparse(adata.layers["counts"]),
    "counts_layer_dtype": adata.layers.get("counts") is not None and adata.layers["counts"].dtype == np.float32,
    "counts_layer_shape_ok": adata.layers.get("counts") is not None and adata.layers["counts"].shape == adata.X.shape,
    "counts_source_is_raw": adata.uns.get("expression_contract", {}).get("counts_source") == ".raw.X",
    # P1-c: doublet 三态 postconditions（仅在 doublet 检测已运行时验证）
    "doublet_cols_present": (
        all(c in adata.obs for c in ["doublet_score","doublet_class","doublet_include","predicted_doublet"])
    ) if _hd else True,
    "doublet_class_valid": (
        adata.obs["doublet_class"].isin(["singlet","uncertain","doublet"]).all()
    ) if _hd else True,
    "doublet_include_bool": (
        adata.obs["doublet_include"].dtype == bool
    ) if _hd else True,
    "doublet_meta_present": (
        "doublet_contract" in adata.uns
    ) if _hd else True,
}
# P1-c: 用 globals().get 兜底读取 needs_review，避免 doublet 检测未运行时因变量未定义而崩溃
_doublet_needs_review = bool(globals().get("doublet_needs_review", False))
stage_status = determine_stage_status({}, hard_postconditions, allow_no_required_methods=True, needs_review=_doublet_needs_review)
effective_parameters = snapshot_effective_parameters(globals(), exclude=("RSCRIPT_BIN", "R_AVAILABLE"), path_root=Path(_root))
runtime_provenance = collect_runtime_provenance(_root, ("anndata", "scanpy", "numpy", "pandas", "scipy"))
manifest_sha256 = sha256_file(MANIFEST_PATH)
run_paths = prepare_run(RUN_ROOT, RUN_ID)
manifest_payload = {
    "run_id": RUN_ID, "stage": "01_qcd", "stage_status": stage_status.value,
    "source_dataset": str(source_dataset),
    "inputs": [{"path": MANIFEST_PATH, "sha256": manifest_sha256}],
    "effective_parameters": effective_parameters, "runtime_provenance": runtime_provenance,
    "hard_postconditions": hard_postconditions,
}
# P1-c: 仅在 doublet 检测已运行时写入 doublet_contract 元数据，避免 KeyError
if _hd:
    manifest_payload["doublet_contract"] = adata.uns["doublet_contract"]
if stage_status.value == "FAILED":
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise RuntimeError(f"Stage 01 FAILED: {hard_postconditions}")

# --- 写入 adata.uns 元数据（FAILED / NEEDS_REVIEW / SUCCESS 共用）---
adata.uns["stage"] = "01_qcd"
adata.uns["status"] = stage_status.value
adata.uns["upstream"] = [MANIFEST_PATH]
adata.uns["version"] = f"v{OUTPUT_VERSION}"
adata.uns["run_id"] = RUN_ID
draft_checkpoint = run_paths.draft_dir / OUTPUT_FILENAME
try:
    adata.write_h5ad(draft_checkpoint, compression="lzf")
    checkpoint_sha256 = sha256_file(draft_checkpoint)
except Exception as error:
    draft_checkpoint.unlink(missing_ok=True)
    manifest_payload["stage_status"] = "FAILED"
    manifest_payload["failure"] = {"type": type(error).__name__, "message": str(error)}
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise
manifest_payload["checkpoint"] = {"path": OUTPUT_FILENAME, "sha256": checkpoint_sha256}

# --- 按 stage_status 分流：NEEDS_REVIEW 只写 draft，SUCCESS 提升 ---
if stage_status.value == "NEEDS_REVIEW":
    # P1-c: needs_review=True 时只写 draft checkpoint，不 promote
    # PI 需要人工查看 doublet_contract.review_reasons 后再决定是调参重跑还是接受
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    OUTPUT_PATH = str(run_paths.draft_dir / OUTPUT_FILENAME)
    print(f"NEEDS_REVIEW 只写 draft {OUTPUT_PATH}  ({adata.n_obs} cells x {adata.n_vars} genes)")
    _dd = adata.uns.get("doublet_contract", {})
    for _r in _dd.get("review_reasons", []):
        print(f"  [需要关注] {_r}")

    # per_dataset schema 校验
    from scrna_integration.per_dataset_schema import validate_per_dataset_output
    _schema_result = validate_per_dataset_output(adata)
    if not _schema_result["passed"]:
        print("per_dataset schema 校验 FAILED:")
        for e in _schema_result["errors"]:
            print(f"  [ERROR] {e}")
    else:
        print("per_dataset schema 校验 PASSED")
    if _schema_result["warnings"]:
        for w in _schema_result["warnings"]:
            print(f"  [WARN] {w}")

    del adata; gc.collect()
    print("内存已释放。")
else:
    # SUCCESS / SUCCESS_WITH_WARNINGS: 正常 promote
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    OUTPUT_PATH = str(promote_run(run_paths))

    # per_dataset schema 校验
    from scrna_integration.per_dataset_schema import validate_per_dataset_output
    _schema_result = validate_per_dataset_output(adata)
    if not _schema_result["passed"]:
        print("per_dataset schema 校验 FAILED:")
        for e in _schema_result["errors"]:
            print(f"  [ERROR] {e}")
    else:
        print("per_dataset schema 校验 PASSED")
    if _schema_result["warnings"]:
        for w in _schema_result["warnings"]:
            print(f"  [WARN] {w}")

    print(f"OK 提升 {OUTPUT_PATH}  ({adata.n_obs} cells x {adata.n_vars} genes)")
    del adata; gc.collect()
    print("内存已释放。")
